# Introduction aux réseaux de neurones avec PyTorch



## Premiers pas avec PyTorch

PyTorch est une bibliothèque d'apprentissage automatique open source développée par Meta, largement utilisée pour la construction et l'entraînement de réseaux de neurones. Elle offre une interface flexible et intuitive pour la définition de modèles à l'aide de tenseurs (tableaux multidimensionnels) et prend en charge la différentiation automatique, ce qui facilite le calcul des gradients pour l'apprentissage. Une caractéristique essentielle de PyTorch est son graphe de calcul dynamique, permettant la modification des modèles à la volée, ce qui est particulièrement utile pour la recherche et l'expérimentation. En robotique, PyTorch est couramment utilisé pour des tâches telles que la perception (par exemple, la détection et la segmentation d'objets à partir de données de caméra), l'estimation d'état et les politiques de contrôle apprises par apprentissage par renforcement, en tirant souvent parti des GPU pour un calcul efficace.

Conversion de NumPy vers Torch et inversement

In [ ]:
import torch
import numpy as np

a_npy = np.eye(4)
print (a_npy)

a_tor = torch.from_numpy(a_npy) # convertir numpy en tenseur torch
print (a_tor)

b_npy = a_tor.numpy() # Convertir Tensor en numpy
print (b_npy)

Notez que si vous modifiez les données dans numpy ou torch, vous les modifiez dans les deux.

In [ ]:
a_npy[0,3] = 99 # Est-ce qu'on modifie uniquement le array numpy ?

a_tor[1,3] = 123 # Est-ce qu'on modifie uniquement le tenseur Torch ?

print (a_npy) # La réponse aux deux questions est NON.
print (a_tor)
print (b_npy)

À part ça, PyTorch est comme NumPy… mais tout porte un nom différent.

In [ ]:
#Ces deux éléments font la même chose, mais portent des noms différents ; l’un est une propriété,
#l’autre est une fonction.

print (a_npy.shape)
print (a_tor.size())

# La plupart des choses que vous pouvez faire avec NumPy (mais pas toutes) sont également possibles avec PyTorch.
# Il vous faudra cependant rechercher leurs noms.

## Réseaux de neurones dans PyTorch

Un réseau de neurones est un modèle informatique inspiré de la structure du cerveau, composé de couches de nœuds interconnectés (ou « neurones ») qui traitent les données en appliquant des poids appris et des transformations non linéaires. Il reçoit une entrée (comme une image ou une donnée de capteur), la fait passer à travers plusieurs couches qui extraient progressivement des caractéristiques de plus en plus abstraites, et produit une sortie telle qu'une classification ou une prédiction. Les réseaux de neurones apprennent à partir des données en ajustant leurs poids pour minimiser l'erreur grâce à des algorithmes comme la rétropropagation, ce qui leur permet de modéliser des relations complexes. En robotique, ils sont couramment utilisés pour les tâches de perception, de prise de décision et de contrôle où la conception de règles traditionnelles est difficile.


Voici comment définir un réseau neuronal dans PyTorch

**IMPORTANT :** Lisez les commentaires. Il est essentiel de bien comprendre le fonctionnement. En cas de doute, posez votre question à votre démonstrateur. Les erreurs simples sont fréquentes avec les réseaux de neurones ; il est donc préférable de demander de l’aide au plus tôt.

In [ ]:
import torch

import torch.nn as nn # fonctions pour les réseaux de neurones, comme les couches et la fonction de perte
import torch.nn.functional as F #  Ce sont des fonctions auxiliaires comme la sigmoïde, la ReLU, etc.

# Vous devez créer une classe qui hérite de nn.Module

class Net(nn.Module):
    def __init__(self):
        # Ici, dans la fonction d'initialisation, vous créez uniquement les calques que vous souhaitez
        # utiliser plus tard. Vous ne les connectez à rien ici.
        # Vous pouvez donc les créer dans n'importe quel ordre.

        super(Net, self).__init__()

        # 3 canaux de couleur en entrée, 6 noyaux de convolution -> 6 canaux en sortie,
        # et également un noyau carré 5x5

        self.conv1 = nn.Conv2d(3, 6, 5)
        # Après application à une image 3x32x32 sans marge ni espacement,
        # le résultat sera de 6x28x28

        # Max pooling avec une fenêtre mobile 2x2
        self.pool = nn.MaxPool2d(2, 2)
        # Après application de ce traitement à l'image 6x28x28, le résultat sera :
        # 6x14x14 (les canaux ne sont pas affectés)

        # 6 canaux d'entrée, 16 noyaux de convolution → 16 canaux de sortie,
        # et également un noyau carré 5x5

        self.conv2 = nn.Conv2d(6, 16, 5)
        # Après application à une image 6x14x14 sans marge intérieure ni espacement,
        # le résultat sera de 16x10x10

        # Plus tard, lors de la passe avant proprement dite, nous appliquerons le maxpooling deux fois,
        # mais nous n'avons besoin de le définir qu'une seule fois, car il ne comporte aucun paramètre
        # que nous rétropropageons.
        # Nous savons donc que nous appliquerons à nouveau MaxPool2d(2,2) à l'image 16x10x10.
        # Par conséquent, la sortie des couches de convolution sera de taille 16x5x5.
        
        # Cette définition de couche nécessite que vous ayez effectué les calculs de convolution.
        # L'« image » finale aura une résolution de 5 × 5 pixels et une profondeur de 16 canaux. Par conséquent,
        # l'entrée est de dimension 16 × 5 × 5.
        self.fc1 = nn.Linear(16 * 5 * 5, 120)

        # La taille de ces couches est _totalement_ arbitraire.
        self.fc2 = nn.Linear(120, 84)

        # Au final, notre sortie sera un vecteur à 10 éléments pour chaque image d'entrée.
        # Ce qui correspond à un encodage one-hot d'un entier de 0 à 9.        
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        # Cette fonction effectue une seule passe directe à travers le réseau.
        # Vous recevez donc l'entrée, la faites passer à travers toutes les couches définies précédemment.
        # (Important : ne définissez aucune nouvelle couche ici) puis vous retournez le résultat final.

        # Appliquer, dans cet ordre : couche convolutionnelle 1 (3,6,5), ReLU, MaxPool2d(2,2)
        x = self.pool(F.relu(self.conv1(x)))

        # Appliquer, dans cet ordre : couche convolutionnelle 2 (6,16,5), ReLU, MaxPool2d(2,2)        
        x = self.pool(F.relu(self.conv2(x)))

        # L'entrée est toujours tridimensionnelle (forme : 16x5x5). Nous la transformons ici :
        # en un vecteur de taille (16*5*5 = 400).        
        x = x.view(-1, 16 * 5 * 5)

        # Faites-le passer par la couche 1 entièrement connectée, puis par une fonction ReLU.
        x = F.relu(self.fc1(x))

        # Faites-le passer par la couche 2 entièrement connectée, puis par une fonction ReLU.
        x = F.relu(self.fc2(x))

        # Faites passer le signal à travers la dernière couche SANS RELU et renvoyez-le.
        x = self.fc3(x)
        return x


# Ici, nous instancions simplement le réseau, afin de pouvoir l'utiliser.
net = Net()

# et assurez-vous qu'il utilise des nombres à virgule flottante 32 bits (« Float »), et non des nombres à virgule flottante 64 bits (« Double »).
net = net.float()


Imprimons le réseau pour voir à quoi il ressemble.

In [ ]:
print (net)

Testez maintenant le réseau avec des entrées aléatoires.

In [ ]:
# créer une seule image (notez que le canal de couleur est au premier plan)
dummy_input = np.random.uniform(low=0, high=1, size=(3,32,32)).astype(np.float32)

# convertir en tenseur :
dummy_tensor = torch.from_numpy(dummy_input)

print ("ancienne taille:", dummy_tensor.size())

# IMPORTANT. Torch fonctionne par lots. Vous pouvez utiliser une taille de lot de 1 (s'il ne s'agit que)
# d'une seule image, mais vous devez ajouter la dimension (qui deviendra le
# nouvel axe 0) :

dummy_tensor = dummy_tensor.unsqueeze(0)

print ("nouvelle taille:", dummy_tensor.size(),"<--- Vous voyez ? Il y a une nouvelle première dimension !")

# Nous pouvons maintenant l'intégrer au réseau :
prediction = net(dummy_tensor)

print ("taille de la prédiction :",prediction.size(),"<-- La sortie a le " \
       "même première dimension. C'est la taille du lot (batch) !")

print ("")
print (prediction)




Commencez à optimiser !

In [ ]:
import torch.optim as optim

# Au lieu de la perte d'erreur quadratique moyenne (MSE ou « L2 »), nous utilisons ici la perte CE car :
# elle offre d'excellentes performances si votre sortie est catégorielle et idéalement comprise dans l'intervalle [0,1].
criterion = nn.CrossEntropyLoss()

# Descente de gradient stochastique... (Il existe de meilleures options)
# et passez net.parameters() à l'optimiseur - ce sont tous les paramètres optimisables
# du réseau
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)

Effectuez un test d'optimisation.

In [ ]:
# Empilons (« concaténer » - « torch.cat() ») 4 images dans un mini-lot:
inputs = torch.cat([dummy_tensor, dummy_tensor, dummy_tensor, dummy_tensor], dim=0).float()
print ("inputs.size()", inputs.size())

# Créons quelques étiquettes factices - remarquez qu'il ne s'agit que de la dimension du lot
labels = torch.zeros(4).long()
print ("labels.size()",labels.size())

# Mettre à zéro les gradients des paramètres (toujours le faire pendant l'apprentissage avant chaque passage avant)
optimizer.zero_grad()

# prédire les résultats
outputs = net(inputs)
print ("outputs.size()", outputs.size())

# Nous rencontrons un problème : nos étiquettes sont unidimensionnelles, mais la prédiction est
# à 10 dimensions (volontairement)... Que faire ?
# Réponse : vous pouvez soit répartir la vérité terrain en un encodage one-hot
# Soit : vous pouvez utiliser une fonction de perte qui accepte les deux : CrossEntropy.

loss = criterion(outputs, labels) # Appliquer la fonction de perte, notez le format :
# loss(prediction, ground_truth) <-- c'est important

# calculer les valeurs de rétropropagation
loss.backward()

# Appliquer les valeurs de rétropropagation selon l'optimiseur
optimizer.step()

# Et la perte d'impression - très important - TRACEZ-LA ! Si elle ne diminue pas beaucoup,
# alors vous avez terminé et le réseau a convergé
print ("Loss:", loss.item())

# Maintenant, exécutez cette cellule plusieurs fois et observez la perte diminuer.

## Entraînons-nous sur des données réelles

Chargez le jeu de données CIFAR-10.

(Il est très petit et vous pouvez le télécharger directement via torch, sans téléchargement manuel.)

In [ ]:
# torchvision possède des fonctions d'assistance pour les ensembles de données basés sur des images
import torchvision

# Nous voulons appliquer certains effets à toutes nos images ; c’est le rôle des transformations.
import torchvision.transforms as transforms


# Nous souhaitons que toutes nos images entrantes soient converties en tenseurs PyTorch 
# (au lieu d'un tableau NumPy) et normalisées.
# Normalisation : img = (img - moyenne) / écart-type
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))]) # parameters for
# La normalisation correspond à la moyenne et à l'écart type pour chaque canal.


# Téléchargez la partie d'entraînement du jeu de données CIFAR-10 et appliquez les transformations.
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

# Intégrez cela dans un chargeur de données multithread pour un chargement de données multiprocesseur plus rapide
# et pour pouvoir mélanger les données et échantillonner des lots entiers et pas seulement
# des éléments individuels
trainloader = torch.utils.data.DataLoader(trainset, batch_size=4,
                                          shuffle=True, num_workers=0)

# même chose pour l'ensemble de données de test

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=4,
                                         shuffle=False, num_workers=0)

# ce sont les noms des étiquettes, correspondant à 0, 1, 2, 3, etc.
classes = ('plane', 'car', 'bird', 'cat',
           'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

Vérifiez TOUJOURS vos données !

Avant de réaliser _n'importe quelle_ expérience, examinez toujours vos données et assurez-vous qu'elles correspondent à vos attentes.

In [ ]:
print ("dataset length:",len(trainset))

image, label = trainset[0]

print ("single image (size):",image.size()) # <-- Il s'agit d'un tenseur
print ("single label:",label) # <-- Ce n'est pas, mais ça le sera si nous utilisons le « trainloader » mentionné plus haut.


# Maintenant, regardons les images.

import matplotlib.pyplot as plt

def imshow(img):
    img = img / 2 + 0.5     # "dénormalisation"
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))


# Récupérez des images d'entraînement aléatoires à l'aide de trainloader
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Afficher les images - créer une grille de 4
imshow(torchvision.utils.make_grid(images))

# étiquettes imprimées
print(' '.join('%5s' % classes[labels[j]] for j in range(4)))



In [ ]:
print (images.min(), images.mean(), images.max()) # il est important de vérifier votre distribution

## En rassemblant tous les éléments

Exécutons l'optimisation en boucle avec l'ensemble de données.

In [ ]:
print ("[epoch, line of data]")

for epoch in range(2):  # parcourir l'ensemble de données plusieurs fois

    running_loss = 0.0

    for i, data in enumerate(trainloader, 0):

        # obtenir les entrées
        inputs, labels = data

        # annuler les gradients des paramètres
        optimizer.zero_grad()

        # aller en avant + retour en arrière + optimiser
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        # imprimer les statistiques
        running_loss += loss.item()

        if i % 2000 == 1999:    # imprimer tous les 2000 mini-lots
            print('[{}, {}] \tloss: {}'.format(
                epoch + 1,
                i + 1,
                running_loss / 2000)
                )

            running_loss = 0.0

print('Entrainement terminée')


Vous pouvez exécuter la cellule ci-dessous pour examiner les résultats. Ils ne sont pas très bons, n'est-ce pas ?

Exécutez à nouveau la cellule précédente. On constate que la perte continue de diminuer. Exécutez à nouveau la cellule ci-dessous. L'amélioration se poursuit-elle ?

(Bien sûr, dans un contexte réel, vous utiliseriez une boucle plutôt que de copier la cellule.)

Inspection – vérifiez manuellement vos prédictions

In [ ]:
dataiter = iter(testloader)
images, labels = next(dataiter)

# imprimer les images
imshow(torchvision.utils.make_grid(images))

print('GroundTruth: ', ' '.join('%5s' % classes[labels[j]] for j in range(4)))

# Passons maintenant aux prédictions du modèle.
outputs = net(images)

# argmax les 10 dimensions en une seule
_, predicted = torch.max(outputs, 1)

# obtenir les noms des étiquettes pour chaque étiquette entière
print('Predicted:    ', ' '.join('%5s' % classes[predicted[j]]
                              for j in range(4)))

Obtenez une évaluation numérique.

In [ ]:
correct = 0
total = 0

# Si nous n'avons rien à apprendre, nous utilisons l'environnement torch.no_grad().
# Dans cet environnement, aucun gradient n'est calculé.

with torch.no_grad():
    for data in testloader:

        images, labels = data
        outputs = net(images)

        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print("Exactitude du réseau sur le " \
      "10000 images de test: {}%".format(100 * correct / total))

Un résultat d'environ 60 % n'est pas très précis. Vous pouvez faire beaucoup mieux.

Voici quelques pistes d'amélioration :

- Essayez un autre optimiseur.

- Augmentez le nombre d'époques d'entraînement.

- Utilisez un réseau de neurones plus grand (plus de nœuds cachés ou plus de couches).

- Visualisez l'évolution de la perte au fil du temps et arrêtez l'entraînement lorsque la perte converge.

- (Si vous le souhaitez) Essayez l'augmentation de données : appliquez des transformations à vos images avant de les transmettre au réseau (rotation aléatoire, bruit aléatoire, etc.). Voici la liste des transformations disponibles : https://pytorch.org/docs/stable/torchvision/transforms.html

# Contributeurs / Contributrices

 - [Florian Golemo](https://fgolemo.github.io/)
 - [Bhairav Mehta](https://bhairavmehta95.github.io/)
 - [Charlie Gauthier](https://velythyl.github.io/)
